In [1]:
#! pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb openai tiktoken unstructured requests beautifulsoup4 lxml


In [2]:
from langchain_core.documents import Document


In [3]:
from langchain_community.document_loaders import TextLoader
loader= TextLoader("anythin.txt",encoding="utf-8")


C:\Users\shaik zabiulla\AppData\Local\Temp\ipykernel_17844\2861631158.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
documents=loader.load()

In [5]:
documents

[Document(metadata={'source': 'anythin.txt'}, page_content="Python is a high-level, interpreted programming language known for its simple syntax, readability, and versatile ecosystem. It is widely used in data science, artificial intelligence, web development, automation, and backend engineering. Because it is highly dynamic and has robust library support, Python is the primary language used to build Retrieval-Augmented Generation (RAG) pipelines.\n\nWhy Python is Used for RAG:\n\nOrchestration Frameworks: Python hosts the primary frameworks for RAG development, such as LangChain, LlamaIndex, and Haystack. These libraries orchestrate the pipeline from document loading to vector search and LLM prompting.\n\nVector Database SDKs: All major vector databases, including Pinecone, Milvus, Chroma, Qdrant, and Weaviate, provide first-class Python SDKs for indexing and querying vector embeddings.\n\nMachine Learning Ecosystem: Python is the native language for PyTorch, TensorFlow, and Hugging F

In [6]:
import os 
from langchain_community.document_loaders import PyPDFLoader


In [7]:
def load_all_pdfs():
    folder_path="C:/Users/shaik zabiulla/Desktop/rag/DATA"
    num_docs=0
    all_documents=[]
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path=os.path.join(folder_path,filename)
            loader=PyPDFLoader(file_path)
            documents=loader.load()
            all_documents.extend(documents)
            num_docs+=1
    print("total pdfs:",num_docs)
    print("total pages:",len(all_documents))
    return all_documents
   





In [8]:
all_documents=load_all_pdfs()

total pdfs: 2
total pages: 29


In [9]:
#chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_document(documents,chunk_size=500,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunked_docs=text_splitter.split_documents(documents)
    return chunked_docs
    



In [10]:
chunks=split_document(all_documents)

In [11]:
from sentence_transformers import SentenceTransformer



In [12]:
class embedding_manager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("loading model:",self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("embedding dimensions=",self.model.get_sentence_embedding_dimension())
    

    def generate_embedings(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("embeddings shape",embeddings.shape)
        return embeddings
        

In [13]:
embedding_manager=embedding_manager()

loading model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


C:\Users\shaik zabiulla\AppData\Local\Temp\ipykernel_17844\3139414744.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=",self.model.get_sentence_embedding_dimension())


In [14]:
import chromadb
import uuid

In [39]:

class vector_store_manager:
    def __init__(self,presist_directorty="DATA/vectorstore",collection_name="pdf_documents"):
        self.collection_name=collection_name
        self.presist_directory=presist_directorty
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.presist_directory,exist_ok=True)
        self.client=chromadb.PersistentClient(path=self.presist_directory)
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"vector store for pdf embeddings in rag"}

        )
        print("initilaizedd the vector store with collection",self.collection_name)
        print("docs in collection", self.collection.count())


    def store_documents(self,documents,embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must be the same.")
        

        ids=[]
        all_metadata=[]
        documents_content=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id=f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_length"]=len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )
        print("total documents added in vector store",len(documents_content))
        print("docs in collection",self.collection.count())

            
    



  

    

In [40]:
vector_store=vector_store_manager()

initilaizedd the vector store with collection pdf_documents
docs in collection 347


In [41]:
texts=[doc.page_content for doc in chunks]

embeddings=embedding_manager.generate_embedings(texts)

vector_store.store_documents(chunks,embeddings)


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape (347, 384)
total documents added in vector store 347
docs in collection 694


In [42]:
from sklearn.metrics.pairwise import cosine_similarity

In [97]:
class RAGretriver:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager=embedding_manager
        self.vector_store=vector_store

    def retrive (self,query,top_k=5,score_threshold=0.1):

        query_embedding=self.embedding_manager.generate_embedings([query])[0]
        results=self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,

        )
        retrived_docs=[]
        if results ["documents"] and results ["documents"][0]:
            ids=results["ids"][0]
            metadatas=results["metadatas"][0]
            documents=results["documents"][0]
            distances=results["distances"][0]
            for i ,(doc_id,metadata,document,distance) in enumerate (zip(ids,metadatas,documents,distances)):
                similarity_score=1-distance
                if similarity_score>=score_threshold:
                    retrived_docs.append(
                        {
                            "doc_id":doc_id,
                            "metadata":metadata,
                            "distance":distance,
                            "document":document,
                            "similarity_score":similarity_score,
                            "rank":i+1
                            }
                    )
            print("retrived documents:",len(retrived_docs))
                    

        else:
            print("no documents found")
        return retrived_docs



In [99]:
rag_retriver=RAGretriver(embedding_manager,vector_store)


In [103]:

rag_retriver.retrive("what is rag retriver")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape (1, 384)
retrived documents: 5


[{'doc_id': 'doc_727329b7-32ec-49b9-aaf1-9499300a3ade',
  'metadata': {'author': '',
   'page': 0,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'total_pages': 21,
   'producer': 'pdfTeX-1.40.25',
   'trapped': '/False',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'moddate': '2024-03-28T00:54:45+00:00',
   'source': 'C:/Users/shaik zabiulla/Desktop/rag/DATA\\research2.pdf',
   'content_length': 288,
   'subject': '',
   'keywords': '',
   'creator': 'LaTeX with hyperref',
   'doc_index': 114,
   'title': '',
   'page_label': '1'},
  'distance': 0.720841109752655,
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'similarity_score': 0.27915889024734497,


In [107]:
from langchain_groq import ChatGroq

In [116]:
llm=ChatGroq(
    api_key=API_KEY_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.7,
    max_tokens=512

)

In [117]:
def generate_answer(query,retriver,llm,top_k=3):
    results=retriver.retrive(query,top_k=top_k)
    context="\n".join([doc["document"]for doc in results]) if results else""
    if not context:
        print("we found no relevat context to generate answer for query ")


    prompt=f""" use given context to generate answer for the query 
    context:{context}
query:{query}
"""

    responses=llm.invoke([prompt.format(context=context,query=query)])
    return responses.content

In [122]:
answer=generate_answer("what is sociala support",rag_retriver,llm)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape (1, 384)
retrived documents: 3


In [123]:
print(answer)

<think>
Okay, the user is asking "what is social support." Let me start by looking at the provided context. The context mentions that older people's support networks are measured by a combination of what they receive and provide. It refers to Antonucci (1985) and House et al. (1988). 

First, I need to define social support based on the context. The key points are that it's not just about receiving support but also about giving it. There's an emphasis on the reciprocal nature of support in close relationships. The context also warns against assuming that all benefits of social contact come solely from the supportive quality, suggesting that giving support might have its own effects, possibly on mortality risk.

I should make sure to include both giving and receiving in the definition. Also, mention the correlation between receiving support and other relationship aspects. The context talks about the need for further research into the assumption that the benefits are due to the supportiv